In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [5]:
def preprocess(df):
    # 할인건수 전처리: "회 이상" 제거 후 숫자 변환
    count_cols2 = ['할인건수_R3M', '할인건수_B0M']
    for col in count_cols2:
        df[col] = (
            df[col].astype(str)
                  .str.replace('회 이상', '', regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 정수 인코딩할 카테고리형 변수들
    le_cols = ['대표결제방법코드', '대표청구지고객주소구분코드', '대표청구서수령지구분코드', '청구서수령방법']
    for col in le_cols:
        codes, uniques = pd.factorize(df[col], sort=True)
        df[col] = codes

    return df

df1 = preprocess(pd.read_parquet('train/4.청구입금정보/201807_train_청구정보.parquet'))
df2 = preprocess(pd.read_parquet('train/4.청구입금정보/201808_train_청구정보.parquet'))
df3 = preprocess(pd.read_parquet('train/4.청구입금정보/201809_train_청구정보.parquet'))
df4 = preprocess(pd.read_parquet('train/4.청구입금정보/201810_train_청구정보.parquet'))
df5 = preprocess(pd.read_parquet('train/4.청구입금정보/201811_train_청구정보.parquet'))
df6 = preprocess(pd.read_parquet('train/4.청구입금정보/201812_train_청구정보.parquet'))

In [6]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

,ID,혜택수혜금액_R3M,선결제건수_R3M,할인금액_R3M,마일_이용포인트_R3M,연체건수_R3M,마일_이용포인트_R12M,연체건수_R6M,청구서수령방법,청구서발송여부_B0,...,포인트_마일리지_건별_R3M,포인트_이용포인트_R3M,청구서발송여부_R3M,포인트_이용포인트_R12M,마일_적립포인트_R12M,마일_적립포인트_R3M,청구금액_B0,할인건수_B0M,할인금액_청구서_R3M,상환개월수_결제일_R3M
0,TRAIN_000000,0.18750,0.0,0.0,0.0,0.0,0.0,0.0625,2.0,1.0,...,0.0,943.21875,1.00000,2457.90625,0.0,0.0,14113.56250,1.0,0.00000,3.00000
1,TRAIN_000001,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,1.0,...,0.0,0.00000,1.00000,0.00000,0.0,0.0,2880.84375,1.0,478.34375,3.00000
2,TRAIN_000002,125.31250,0.0,0.0,0.0,0.0,0.0,0.0000,5.0,1.0,...,0.0,2372.90625,1.00000,4587.40625,0.0,0.0,28629.87500,1.0,0.00000,3.00000
3,TRAIN_000003,61.06250,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,1.0,...,0.0,4217.93750,1.00000,12049.81250,0.0,0.0,23646.81250,1.0,0.00000,3.00000
4,TRAIN_000004,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,0.0,0.00000,0.03125,0.00000,0.0,0.0,0.00000,1.0,0.00000,0.03125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,0.0,1972.71875,0.00000,5786.78125,0.0,0.0,0.00000,1.0,0.00000,0.00000
399996,TRAIN_399996,216.46875,0.0,0.0,0.0,0.0,0.0,0.0000,5.0,1.0,...,0.0,15703.50000,1.00000,83377.56250,0.0,0.0,13804.78125,1.0,209.50000,3.00000
399997,TRAIN_399997,0.00000,0.0,0.0,0.0,0.0,0.0,0.0625,4.0,1.0,...,0.0,0.00000,1.00000,0.00000,0.0,0.0,6898.15625,1.0,0.00000,3.00000
399998,TRAIN_399998,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.00000,1.0,0.00000,0.00000


In [7]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,혜택수혜금액_R3M,선결제건수_R3M,할인금액_R3M,마일_이용포인트_R3M,연체건수_R3M,마일_이용포인트_R12M,연체건수_R6M,청구서수령방법,청구서발송여부_B0,...,포인트_이용포인트_R3M,청구서발송여부_R3M,포인트_이용포인트_R12M,마일_적립포인트_R12M,마일_적립포인트_R3M,청구금액_B0,할인건수_B0M,할인금액_청구서_R3M,상환개월수_결제일_R3M,Segment
0,TRAIN_000000,0.18750,0.0,0.0,0.0,0.0,0.0,0.0625,2.0,1.0,...,943.21875,1.00000,2457.90625,0.0,0.0,14113.56250,1.0,0.00000,3.00000,D
1,TRAIN_000001,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,1.0,...,0.00000,1.00000,0.00000,0.0,0.0,2880.84375,1.0,478.34375,3.00000,E
2,TRAIN_000002,125.31250,0.0,0.0,0.0,0.0,0.0,0.0000,5.0,1.0,...,2372.90625,1.00000,4587.40625,0.0,0.0,28629.87500,1.0,0.00000,3.00000,C
3,TRAIN_000003,61.06250,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,1.0,...,4217.93750,1.00000,12049.81250,0.0,0.0,23646.81250,1.0,0.00000,3.00000,D
4,TRAIN_000004,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,0.00000,0.03125,0.00000,0.0,0.0,0.00000,1.0,0.00000,0.03125,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,1972.71875,0.00000,5786.78125,0.0,0.0,0.00000,1.0,0.00000,0.00000,E
399996,TRAIN_399996,216.46875,0.0,0.0,0.0,0.0,0.0,0.0000,5.0,1.0,...,15703.50000,1.00000,83377.56250,0.0,0.0,13804.78125,1.0,209.50000,3.00000,D
399997,TRAIN_399997,0.00000,0.0,0.0,0.0,0.0,0.0,0.0625,4.0,1.0,...,0.00000,1.00000,0.00000,0.0,0.0,6898.15625,1.0,0.00000,3.00000,C
399998,TRAIN_399998,0.00000,0.0,0.0,0.0,0.0,0.0,0.0000,4.0,0.0,...,0.00000,0.00000,0.00000,0.0,0.0,0.00000,1.0,0.00000,0.00000,E


In [8]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]


In [9]:
ex1 = merged_df

In [10]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['선결제건수_R3M', '마일_이용포인트_R3M', '연체건수_R3M', '마일_이용포인트_R12M', '연체건수_R6M', '포인트_마일리지_월적립_R3M', '선결제건수_R6M', '할인건수_R3M', '포인트_적립포인트_R3M', '대표결제방법코드', '포인트_포인트_월적립_B0M', '포인트_마일리지_환산_B0M', '포인트_포인트_월적립_R3M', '포인트_포인트_건별_B0M', '마일_잔여포인트_B0M', '포인트_마일리지_월적립_B0M', '포인트_마일리지_건별_B0M', '포인트_잔여포인트_B0M', '포인트_포인트_건별_R3M', '포인트_마일리지_건별_R3M', '마일_적립포인트_R12M', '마일_적립포인트_R3M', '할인건수_B0M']


In [11]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 18


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,청구서수령방법,대표청구서수령지구분코드,0.999982,0.028652,0.028315
1,할인금액_R3M,할인금액_B0M,0.996026,0.185358,0.184801
2,혜택수혜금액_R3M,혜택수혜금액,0.994624,0.361241,0.360293
3,할인금액_청구서_B0M,할인금액_청구서_R3M,0.994507,0.203588,0.203312
4,청구금액_R3M,청구금액_B0,0.990861,0.603846,0.598621
5,청구금액_R3M,청구금액_R6M,0.986785,0.603846,0.609427
6,청구서발송여부_B0,청구서발송여부_R3M,0.980884,0.215284,0.210494
7,상환개월수_결제일_R6M,상환개월수_결제일_R3M,0.974951,0.141026,0.132145
8,청구금액_R6M,청구금액_B0,0.972458,0.609427,0.598621
9,포인트_이용포인트_R3M,포인트_이용포인트_R12M,0.937914,0.172498,0.143066


In [12]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 11
제거할 피처 목록:
['할인금액_B0M', '청구금액_R3M', '할인금액_R3M', '대표청구서수령지구분코드', '청구서발송여부_R3M', '포인트_이용포인트_R12M', '청구금액_B0', '청구서발송여부_R6M', '할인금액_청구서_R3M', '상환개월수_결제일_R3M', '혜택수혜금액']


In [13]:
cols_to_drop = ['할인금액_B0M', '청구금액_R3M', '할인금액_R3M', '대표청구서수령지구분코드', '청구서발송여부_R3M', '포인트_이용포인트_R12M', '청구금액_B0', '청구서발송여부_R6M', '할인금액_청구서_R3M', '상환개월수_결제일_R3M', '혜택수혜금액']
ex1.drop(columns=cols_to_drop, inplace=True)

In [14]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 '혜택수혜금액_R3M',
 '청구서수령방법',
 '청구서발송여부_B0',
 '대표청구지고객주소구분코드',
 '대표결제일',
 '청구금액_R6M',
 '할인금액_청구서_B0M',
 '포인트_적립포인트_R12M',
 '상환개월수_결제일_R6M',
 '포인트_이용포인트_R3M',
 'Segment']

In [15]:
ex1.to_parquet('청구_전처리_Segment.parquet', index=False)

In [16]:
ddf1 = preprocess(pd.read_parquet('train/4.청구입금정보/201807_train_청구정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/4.청구입금정보/201808_train_청구정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/4.청구입금정보/201809_train_청구정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/4.청구입금정보/201810_train_청구정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/4.청구입금정보/201811_train_청구정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/4.청구입금정보/201812_train_청구정보.parquet'))

In [17]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

             ID      대표결제일  대표결제방법코드  대표청구지고객주소구분코드  대표청구서수령지구분코드  청구서수령방법  \
0  TRAIN_000000  27.000000       0.0            0.0      2.000000      2.0   
1  TRAIN_000001  13.000000       0.0            1.0      4.038462      4.0   
2  TRAIN_000002  11.576923       0.0            0.0      5.038462      5.0   
3  TRAIN_000003   5.000000       0.0            1.0      4.038462      4.0   
4  TRAIN_000004  13.000000       0.0            1.0      4.038462      4.0   

   청구서발송여부_B0  청구서발송여부_R3M  청구서발송여부_R6M       청구금액_B0  ...  할인금액_청구서_B0M  \
0         1.0     1.000000     1.000000  14633.500000  ...      0.000000   
1         1.0     1.000000     1.000000   3142.653846  ...    163.384615   
2         1.0     1.000000     1.000000  26756.192308  ...      0.000000   
3         1.0     1.000000     1.000000  21448.692308  ...      0.000000   
4         0.0     0.038462     0.615385      0.000000  ...      0.000000   

   상환개월수_결제일_R6M  상환개월수_결제일_R3M  선결제건수_R6M  선결제건수_R3M  연체건수_R6M  연체건수_R3M 

In [18]:
cols = ['ID',
 '혜택수혜금액_R3M',
 '청구서수령방법',
 '청구서발송여부_B0',
 '대표청구지고객주소구분코드',
 '대표결제일',
 '청구금액_R6M',
 '할인금액_청구서_B0M',
 '포인트_적립포인트_R12M',
 '상환개월수_결제일_R6M',
 '포인트_이용포인트_R3M',]

result = result[cols]
result

,ID,혜택수혜금액_R3M,청구서수령방법,청구서발송여부_B0,대표청구지고객주소구분코드,대표결제일,청구금액_R6M,할인금액_청구서_B0M,포인트_적립포인트_R12M,상환개월수_결제일_R6M,포인트_이용포인트_R3M
0,TRAIN_000000,0.692308,2.0,1.0,0.0,27.000000,104781.461538,0.000000,3511.153846,5.769231,979.846154
1,TRAIN_000001,0.000000,4.0,1.0,1.0,13.000000,18480.423077,163.384615,0.000000,6.000000,0.000000
2,TRAIN_000002,132.730769,5.0,1.0,0.0,11.576923,160012.538462,0.000000,13702.115385,6.000000,2498.846154
3,TRAIN_000003,30.500000,4.0,1.0,1.0,5.000000,122201.961538,0.000000,11324.576923,5.769231,3755.269231
4,TRAIN_000004,0.000000,4.0,0.0,1.0,13.000000,112.500000,0.000000,0.000000,0.807692,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.000000,4.0,0.0,1.0,25.000000,0.000000,0.000000,2284.230769,0.000000,1898.500000
399996,TRAIN_399996,246.076923,5.0,1.0,0.0,20.000000,144308.269231,73.115385,61598.038462,6.000000,17410.807692
399997,TRAIN_399997,0.000000,4.0,1.0,2.0,20.000000,42036.846154,0.000000,0.000000,5.769231,0.000000
399998,TRAIN_399998,0.000000,4.0,0.0,1.0,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [19]:
result.to_parquet('청구_전처리_test.parquet', index=False)

In [20]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
